In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.
# This source code is licensed under the CC-by-NC licence,
# found in the LICENSE_CELL_DINO_CODE file in the root directory of this source tree.

In [ ]:
SAMPLE_IMAGES_DIR = "sample_images/"  # path to directory with cell images.
REPO_DIR="" # path to the dinov2 repo.
# Also define the urls of the pretrained models CPurl, SCurl, FOVurl used in the next cell. Instructions to get the models urls are in the README.md.

In [ ]:
import torch
cell_dino_vits8 = torch.hub.load(REPO_DIR, 'cell_dino_cp_vits8', source='local', pretrained_url=CPurl)
cell_dino_vitl16_sc = torch.hub.load(REPO_DIR, 'cell_dino_hpa_vitl16', source='local', pretrained_url=SCurl)
cell_dino_vitl16_fov = torch.hub.load(REPO_DIR, 'cell_dino_hpa_vitl16', source='local', pretrained_url=FOVurl)
#channel_adaptive_dino_vitl16 = torch.hub.load(REPO_DIR, 'channel_adaptive_dino_vitl16', source='local', pretrained_url=CAurl)
# cell_dino_vitl14 = torch.hub.load(REPO_DIR, 'cell_dino_hpa_vitl14', source='local', pretrained_url=HRurl)

In [ ]:
import torch
import torchvision
from dinov2.hub.cell_dino.backbones import cell_dino_hpa_vitl16, cell_dino_cp_vits8
from functools import partial
from dinov2.eval.utils import ModelWithIntermediateLayers

# Use GPU when available, otherwise safely fall back to CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class self_normalize(object):
    def __call__(self, x):
        x = x.float()
        x_max = float(x.max().item()) if x.numel() > 0 else 1.0
        if x_max > 255.0:
            x = x / 65535.0
        elif x_max > 1.0:
            x = x / 255.0
        m = x.mean((-2, -1), keepdim=True)
        s = x.std((-2, -1), unbiased=False, keepdim=True)
        x -= m
        x /= s + 1e-7
        return x

normalize = self_normalize()

In [ ]:
# ---------------------- Example inference on HPA-FoV dataset --------------------------

# 1- Read one human protein atlas HPA-FoV image (4 channels)
img = torchvision.io.read_image(SAMPLE_IMAGES_DIR + "HPA_FoV_00070df0-bbc3-11e8-b2bc-ac1f6b6435d0.png")

# 2- Normalise image as it was done for training
img_hpa_fov = img.unsqueeze(0).to(device=DEVICE)
img_hpa_fov = normalize(img_hpa_fov)

# 3- Load model
cell_dino_model = cell_dino_vitl16_fov
cell_dino_model.to(device=DEVICE)
cell_dino_model.eval()

# 4- Inference
features = cell_dino_model(img_hpa_fov)
print(features)

# 5- [Optional] feature extractor as used for linear evaluation
autocast_ctx = partial(torch.cuda.amp.autocast, enabled=True, dtype=torch.float)
model_with_interm_layers = ModelWithIntermediateLayers(cell_dino_model, 4, autocast_ctx)
features_with_interm_layers = model_with_interm_layers(img_hpa_fov)

In [ ]:
# ---------------------- Example inference on cell painting data --------------------------

# 1- Read one cell painting image (5 channels)
img = torchvision.io.read_image(SAMPLE_IMAGES_DIR + "CP_BBBC036_24277_a06_1_976@140x149.png")
img5_channels = torch.zeros([1, 5, 160, 160])
for c in range(5):
    img5_channels[0, c] = img[0, :, 160 * c : 160 * (c + 1)]
img5_channels = img5_channels.to(device=DEVICE)

# 2- Normalise image as it was done for training
img5_channels = normalize(img5_channels)

# 3- Load model
cell_dino_model = cell_dino_vits8
cell_dino_model.to(device=DEVICE)
cell_dino_model.eval()

# 4- Inference
features = cell_dino_model(img5_channels)
print(features[0,0:10])

In [ ]:
# ---------------------- Example inference on HPA single cell dataset --------------------------

# Read one human protein atlas HPA single cell image (4 channels)
img = torchvision.io.read_image(SAMPLE_IMAGES_DIR + "HPA_single_cell_00285ce4-bba0-11e8-b2b9-ac1f6b6435d0_15.png")

# 2- Normalise image as it was done for training
img_hpa = img.unsqueeze(0).to(device=DEVICE)
img_hpa = normalize(img_hpa)

# 3- Load model
cell_dino_model = cell_dino_vitl16_sc
cell_dino_model.to(device=DEVICE)
cell_dino_model.eval()

# 4- Inference
features = cell_dino_model(img_hpa)
print(features)

torch.save(features.cpu(), "sample_features_hpa.pt")

In [ ]:
# ---------------------- Batch inference on a custom 4-channel NumPy dataset --------------------------
from pathlib import Path
import numpy as np

NPY_PATH = Path("/scratch/svo/morpho_analysis/toy_dataset/MIPFTMA14/aveolar/processed_img_with_mask.npy")
OUTPUT_FEATURES_PATH = Path("/path/to/output/custom_4ch_128_features.pt")
BATCH_SIZE = 32
TARGET_SIZE = (128, 128)

# Pick one of the 4-channel pretrained checkpoints (single-cell or FoV).
cell_dino_model = cell_dino_vitl16_sc
cell_dino_model.to(device=DEVICE)
cell_dino_model.eval()
model_device = next(cell_dino_model.parameters()).device

images_np = np.load(NPY_PATH, mmap_mode="r")
if images_np.ndim != 4:
    raise ValueError(f"Expected shape [N, C, H, W], got {images_np.shape}")
if images_np.shape[1] != 4:
    raise ValueError(f"Expected 4 channels, got {images_np.shape[1]}")

num_samples = images_np.shape[0]
print(f"Loaded {num_samples} samples from {NPY_PATH}")
print(f"Input dtype: {images_np.dtype}, value range: [{images_np.min()}, {images_np.max()}]")
print(f"Running inference on device: {model_device}")

all_features = []

with torch.no_grad():
    for start in range(0, num_samples, BATCH_SIZE):
        end = min(start + BATCH_SIZE, num_samples)
        batch_np = np.asarray(images_np[start:end])
        batch = torch.from_numpy(batch_np).to(device=model_device, dtype=torch.float32)

        if tuple(batch.shape[-2:]) != TARGET_SIZE:
            batch = torch.nn.functional.interpolate(
                batch, size=TARGET_SIZE, mode="bilinear", align_corners=False
            )

        batch = normalize(batch)
        features = cell_dino_model(batch)
        all_features.append(features.cpu())

features_tensor = torch.cat(all_features, dim=0)
OUTPUT_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {"indices": list(range(num_samples)), "features": features_tensor},
    OUTPUT_FEATURES_PATH,
)

print(f"Saved features for {features_tensor.shape[0]} samples to {OUTPUT_FEATURES_PATH}")
print(f"Feature tensor shape: {features_tensor.shape}")

In [ ]:
# ---------------------- Supervised setup from NumPy arrays (.npy) --------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

IMAGES_NPY = Path("/scratch/svo/morpho_analysis/toy_dataset/MIPFTMA14/aveolar/processed_img_with_mask.npy")
LABELS_NPY = None  # e.g. Path("/path/to/labels.npy")
LABELS_CSV = None  # e.g. Path("/path/to/labels.csv")
LABEL_COL = "label"
VAL_SPLIT = 0.2
RANDOM_SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 4
TARGET_SIZE = (128, 128)

images_np = np.load(IMAGES_NPY, mmap_mode="r")
if images_np.ndim != 4:
    raise ValueError(f"Expected shape [N, C, H, W], got {images_np.shape}")
if images_np.shape[1] != 4:
    raise ValueError(f"Expected 4 channels, got {images_np.shape[1]}")
num_samples = int(images_np.shape[0])
indices = np.arange(num_samples)

labels_raw = None
if LABELS_NPY is not None:
    labels_raw = np.load(LABELS_NPY)
    if labels_raw.shape[0] != num_samples:
        raise ValueError("LABELS_NPY length must match number of samples")
elif LABELS_CSV is not None:
    labels_df = pd.read_csv(LABELS_CSV)
    if LABEL_COL not in labels_df.columns:
        raise ValueError(f"LABEL_COL '{LABEL_COL}' missing in LABELS_CSV")
    if len(labels_df) != num_samples:
        raise ValueError("LABELS_CSV row count must match number of samples")
    labels_raw = labels_df[LABEL_COL].to_numpy()

HAS_LABELS = labels_raw is not None
label_to_id = {}
id_to_label = {}
label_ids = None
label_names = []

if HAS_LABELS:
    labels_series = pd.Series(labels_raw).astype(str)
    label_names = sorted(labels_series.unique().tolist())
    label_to_id = {name: idx for idx, name in enumerate(label_names)}
    id_to_label = {idx: name for name, idx in label_to_id.items()}
    label_ids = labels_series.map(label_to_id).to_numpy(dtype=np.int64)

class FourChannelNpyDataset(Dataset):
    def __init__(self, images_array, sample_indices, labels_array, target_size):
        self.images_array = images_array
        self.sample_indices = np.asarray(sample_indices, dtype=np.int64)
        self.labels_array = labels_array
        self.target_size = target_size

    def __len__(self):
        return len(self.sample_indices)

    def __getitem__(self, idx):
        sample_idx = int(self.sample_indices[idx])
        img_np = np.asarray(self.images_array[sample_idx])  # C x H x W
        if img_np.shape[0] != 4:
            raise ValueError(f"Expected 4 channels, got {img_np.shape[0]} at index {sample_idx}")

        img = torch.from_numpy(img_np).float()
        if tuple(img.shape[-2:]) != self.target_size:
            img = torch.nn.functional.interpolate(
                img.unsqueeze(0),
                size=self.target_size,
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

        img = normalize(img.unsqueeze(0)).squeeze(0)

        if self.labels_array is None:
            label = -1
        else:
            label = int(self.labels_array[sample_idx])
        return img, label, sample_idx

if HAS_LABELS:
    stratify = label_ids if len(label_names) > 1 else None
    train_idx, val_idx = train_test_split(
        indices,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=stratify,
    )
else:
    train_idx, val_idx = indices, np.array([], dtype=np.int64)

train_ds = FourChannelNpyDataset(images_np, train_idx, label_ids, TARGET_SIZE)
val_ds = FourChannelNpyDataset(images_np, val_idx, label_ids, TARGET_SIZE) if len(val_idx) > 0 else None

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = (
    DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    if val_ds is not None
    else None
)

print(f"Loaded images: {num_samples}")
print(f"Image dtype: {images_np.dtype}, value range: [{images_np.min()}, {images_np.max()}]")
print(f"Has labels: {HAS_LABELS}")
if HAS_LABELS:
    print(f"Train samples: {len(train_ds)}")
    print(f"Val samples:   {0 if val_ds is None else len(val_ds)}")
    print(f"Num classes:   {len(label_names)}")
else:
    print("Set LABELS_NPY or LABELS_CSV to enable supervised fine-tuning.")

In [ ]:
# ---------------------- Continue training pretrained Cell-DINO for classification --------------------------
import torch.nn as nn

if not HAS_LABELS:
    raise ValueError("Supervised fine-tuning requires labels. Set LABELS_NPY or LABELS_CSV in Cell 9.")
if len(label_names) < 2:
    raise ValueError("Need at least 2 classes for classification fine-tuning.")
if val_loader is None:
    raise ValueError("Validation split is empty. Increase dataset size or adjust VAL_SPLIT.")

EPOCHS = 12
FREEZE_BACKBONE_EPOCHS = 2  # keep backbone frozen for first epochs, then unfreeze
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4

# Choose one pretrained 4-channel backbone.
backbone = cell_dino_vitl16_sc
backbone.to(DEVICE)
model_device = next(backbone.parameters()).device

with torch.no_grad():
    sample_x, _, _ = train_ds[0]
    sample_feat = backbone(sample_x.unsqueeze(0).to(model_device))
    feat_dim = sample_feat.shape[-1]

class CellDinoClassifier(nn.Module):
    def __init__(self, backbone_model, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone_model
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        return self.classifier(feat)

model = CellDinoClassifier(backbone, feat_dim, len(label_names)).to(model_device)

# Start by freezing backbone for stable head warm-up.
for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    [
        {"params": model.classifier.parameters(), "lr": LR_HEAD},
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
    ],
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None

for epoch in range(EPOCHS):
    if epoch == FREEZE_BACKBONE_EPOCHS:
        for p in model.backbone.parameters():
            p.requires_grad = True

    model.train()
    train_correct = 0
    train_total = 0
    train_loss_sum = 0.0

    for xb, yb, _ in train_loader:
        xb = xb.to(model_device, non_blocking=True)
        yb = yb.to(model_device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * yb.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0
    with torch.no_grad():
        for xb, yb, _ in val_loader:
            xb = xb.to(model_device, non_blocking=True)
            yb = yb.to(model_device, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            val_loss_sum += loss.item() * yb.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    train_loss = train_loss_sum / max(train_total, 1)
    val_loss = val_loss_sum / max(val_total, 1)
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

FINETUNED_MODEL_PATH = Path("/path/to/output/cell_dino_finetuned_classifier.pt")
FINETUNED_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
        "best_val_acc": best_val_acc,
        "target_size": TARGET_SIZE,
    },
    FINETUNED_MODEL_PATH,
)
print(f"Best val accuracy: {best_val_acc:.4f}")
print(f"Saved fine-tuned model to: {FINETUNED_MODEL_PATH}")

In [ ]:
# ---------------------- Unsupervised grouping from embeddings (no labels required) --------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

EMBEDDINGS_PATH = Path("/path/to/output/custom_4ch_128_features.pt")
CLUSTERS_CSV_PATH = Path("/path/to/output/embedding_clusters.csv")
N_CLUSTERS = 8
RANDOM_SEED = 42

# Prefer in-memory embeddings from Cell 8; otherwise load from disk.
if "features_tensor" in globals():
    feats = features_tensor.detach().cpu().numpy()
    sample_indices = np.arange(feats.shape[0])
else:
    if not EMBEDDINGS_PATH.exists():
        raise ValueError("Run Cell 8 first or set EMBEDDINGS_PATH to an existing .pt file.")
    payload = torch.load(EMBEDDINGS_PATH, map_location="cpu")
    feats = payload["features"].cpu().numpy()
    sample_indices = np.asarray(payload.get("indices", np.arange(feats.shape[0])))

# L2-normalize embeddings before clustering.
norm = np.linalg.norm(feats, axis=1, keepdims=True)
norm = np.clip(norm, a_min=1e-12, a_max=None)
feats_norm = feats / norm

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init="auto")
cluster_id = kmeans.fit_predict(feats_norm)

cluster_df = pd.DataFrame(
    {
        "sample_index": sample_indices,
        "cluster_id": cluster_id,
    }
)

CLUSTERS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
cluster_df.to_csv(CLUSTERS_CSV_PATH, index=False)

print(f"Computed embeddings for {feats.shape[0]} samples with dim={feats.shape[1]}")
print(f"Assigned {N_CLUSTERS} clusters")
print(f"Saved cluster assignments to: {CLUSTERS_CSV_PATH}")
print(cluster_df["cluster_id"].value_counts().sort_index())